# Single Index Models: Estimation, Uncertainty, and Allocation
This deeper dive develops the mathematical chain behind the L6 examples: regularized SIM estimation, bootstrap uncertainty, covariance construction, uncertainty propagation, and maximum-Sharpe optimization. The notation is reconciled with CHEME 5660's annualized growth-rate observation convention.

> __Learning Objectives:__
>
> * Derive the ridge estimator and its full sandwich covariance.
> * Distinguish empirical-residual and Gaussian parametric bootstraps.
> * Convert consistently between growth-rate, one-step log-return, and diffusion-volatility conventions.
> * Explain how parameter distributions induce portfolio-risk and allocation distributions.
> * Reformulate a maximum-Sharpe ratio problem as a second-order cone program.

___


## 1. SIM Regression and Assumptions
For asset $i$ and market factor $m$,
$$
g_{i,t}=\alpha_i+\beta_i g_{m,t}+\varepsilon_{g,i,t}.
$$
The SIM assumes $\mathbb E[\varepsilon_{g,i,t}]=0$, $\operatorname{Cov}(g_{m,t},\varepsilon_{g,i,t})=0$, and $\operatorname{Cov}(\varepsilon_{g,i,t},\varepsilon_{g,j,t})=0$ for $i\ne j$. These assumptions route contemporaneous cross-asset covariance through the market factor. They do not imply Gaussian tails, temporal independence, or constant future parameters; those are separate modeling choices that must be diagnosed.


## 2. One Model, Three Unit Conventions
The core notebooks observe annualized continuous-growth-rate samples
$$
g_t=\frac{1}{\Delta t}\log\!\left(\frac{P_t}{P_{t-1}}\right).
$$
The fitted residual standard deviation is therefore $\sigma_{g,\varepsilon}=\operatorname{sd}(\varepsilon_g)$, and the covariance of growth-rate observations contains $\sigma_{g,\varepsilon}^2$ directly.

For one-step log returns $r_t=\Delta t\,g_t$,
$$
\boldsymbol\Sigma_r=\Delta t^2\boldsymbol\Sigma_g.
$$
If a diffusion is written as $\operatorname{Var}(r_t)=\Delta t\,\boldsymbol\Sigma_{\mathrm{ann}}$, then
$$
\boldsymbol\Sigma_{\mathrm{ann}}=\Delta t\,\boldsymbol\Sigma_g,
\qquad \sigma_{\mathrm{ann}}=\sqrt{\Delta t}\,\sigma_g.
$$
All three are valid. The error is to mix a market variance from one convention with an idiosyncratic variance from another or to insert an isolated $\Delta t$ into only one term.


## 3. Regularized Estimation and the Ridge Sandwich
Let $\mathbf X=[\mathbf 1\;\mathbf g_m]$, $\mathbf y_i=\mathbf g_i$, and $\boldsymbol\theta_i=(\alpha_i,\beta_i)^\top$. Ridge estimation solves
$$
\hat{\boldsymbol\theta}_i=\arg\min_{\boldsymbol\theta}\left\{\frac12\|\mathbf y_i-\mathbf X\boldsymbol\theta\|_2^2+\frac\delta2\|\boldsymbol\theta\|_2^2\right\},
$$
with solution
$$
\boxed{\hat{\boldsymbol\theta}_i=(\mathbf X^\top\mathbf X+\delta\mathbf I)^{-1}\mathbf X^\top\mathbf y_i}.
$$
Define $\mathbf K^{-1}=(\mathbf X^\top\mathbf X+\delta\mathbf I)^{-1}$ and estimate growth-residual variance by
$$
s_{g,\varepsilon}^2=\frac{\|\mathbf y_i-\mathbf X\hat{\boldsymbol\theta}_i\|_2^2}{T-2}.
$$
The coefficient covariance is the ridge sandwich
$$
\boxed{\widehat{\operatorname{Cov}}(\hat{\boldsymbol\theta}_i)=s_{g,\varepsilon}^2\mathbf K^{-1}\mathbf X^\top\mathbf X\mathbf K^{-1}}.
$$
Only for $\delta=0$ does this reduce to $s_{g,\varepsilon}^2(\mathbf X^\top\mathbf X)^{-1}$. Dropping the middle Gram matrix for nonzero ridge is an algebraic error.


## 4. Bootstrap Uncertainty Quantification
Starting from fitted values $\hat{\mathbf y}=\mathbf X\hat{\boldsymbol\theta}$, each bootstrap replicate constructs $\mathbf y^{(b)}=\hat{\mathbf y}+\boldsymbol\varepsilon^{(b)}$ and refits the model.

- __Empirical-residual bootstrap:__ Sample centered fitted residuals with replacement. This preserves the observed one-step marginal shape, including asymmetry and heavy tails.
- __Gaussian parametric bootstrap:__ Draw $\boldsymbol\varepsilon^{(b)}\sim\mathcal N(\mathbf 0,s_{g,\varepsilon}^2\mathbf I)$. This isolates uncertainty under the Gaussian innovation assumption.

The empirical quantiles of $\{\hat\alpha^{(b)},\hat\beta^{(b)},\hat\sigma_{g,\varepsilon}^{(b)}\}$ give percentile intervals. Comparing the two methods is a model-risk diagnostic. Both procedures are i.i.d.; neither retains the volatility clustering diagnosed in L3a. Moving-block, stationary, or regime-aware bootstraps are required when multi-day persistence is part of the estimand.

For the Gaussian procedure, the limiting covariance is exactly the ridge sandwich above. The [executable L6a notebook](../../CHEME-5660-L6a-Example-SIM-Parameter-Uncertainty-Fall-2026.ipynb) compares empirical and theoretical standard errors.


## 5. SIM Covariance as Rank One Plus Diagonal
Under the SIM orthogonality assumptions, the growth-rate covariance is
$$
\boxed{\boldsymbol\Sigma_g^{\mathrm{SIM}}=\sigma_{g,m}^2\boldsymbol\beta\boldsymbol\beta^\top+\operatorname{diag}(\sigma_{g,\varepsilon,1}^2,\ldots,\sigma_{g,\varepsilon,N}^2)}.
$$
Thus
$$
(\Sigma_g)_{ij}=\beta_i\beta_j\sigma_{g,m}^2\quad(i\ne j),
$$
and
$$
(\Sigma_g)_{ii}=\beta_i^2\sigma_{g,m}^2+\sigma_{g,\varepsilon,i}^2.
$$
A full covariance has $N(N+1)/2$ distinct entries; SIM uses $N$ betas, $N$ idiosyncratic variances, and one market variance. For $N=500$, that is 1,001 rather than 125,250 parameters. The reduction stabilizes estimation but imposes a strong diagonal-residual assumption that must be checked.


## 6. From Parameter Uncertainty to Decision Uncertainty
For each bootstrap scenario $b$, preserve the joint within-asset draw $(\alpha_i^{(b)},\beta_i^{(b)},\sigma_{g,\varepsilon,i}^{(b)})$, construct $(\boldsymbol\mu^{(b)},\boldsymbol\Sigma_g^{(b)})$, and solve for scenario weights $\mathbf w^{(b)}$. Compare them with the deployed point-estimate weights $\hat{\mathbf w}$.

Useful diagnostics include
$$
\text{fixed variance}^{(b)}=\hat{\mathbf w}^\top\boldsymbol\Sigma_g^{(b)}\hat{\mathbf w},
$$
$$
\text{variance regret}^{(b)}=\hat{\mathbf w}^\top\boldsymbol\Sigma_g^{(b)}\hat{\mathbf w}-(\mathbf w^{(b)})^\top\boldsymbol\Sigma_g^{(b)}\mathbf w^{(b)},
$$
and allocation distance $\tfrac12\|\mathbf w^{(b)}-\hat{\mathbf w}\|_1$. This layer matters because optimization is nonlinear: individually narrow coefficient intervals can still generate unstable weights. The [L6b propagation notebook](../../../L6b/CHEME-5660-L6b-Example-SIM-Portfolio-ForwardValidation-Fall-2026.ipynb) implements this workflow.


## 7. Maximum Sharpe Ratio as an SOCP
Let $\mathbf c=\boldsymbol\mu-g_f\mathbf 1$ and factor $\boldsymbol\Sigma=\mathbf U^\top\mathbf U$. The Sharpe ratio is
$$
\operatorname{SR}(\mathbf w)=\frac{\mathbf c^\top\mathbf w}{\|\mathbf U\mathbf w\|_2}.
$$
Using the normalization $\mathbf y=\mathbf w/\tau$, the long-only maximum-Sharpe problem can be written
$$
\min_{\mathbf y,\tau}\;\|\mathbf U\mathbf y\|_2
$$
subject to
$$
\boxed{\mathbf c^\top\mathbf y=1,\qquad \mathbf 1^\top\mathbf y=\tau,\qquad \mathbf y\ge0,\qquad \tau>0}.
$$
The recovered portfolio is $\mathbf w=\mathbf y/\tau$. The norm objective is represented by a second-order cone; the remaining constraints are linear.


## Summary

> __Key Takeaways:__
>
> * __Units are part of the model:__ Convert the complete covariance, not one isolated term.
> * __Ridge uncertainty requires the sandwich:__ The OLS inverse-Gram shortcut does not extend unchanged to $\delta>0$.
> * __Bootstrap method is an assumption:__ Residual and parametric draws provide a useful sensitivity comparison but do not preserve volatility regimes.
> * __SIM is a structural compression:__ Rank-one-plus-diagonal covariance is scalable because it makes strong residual assumptions.
> * __Decision uncertainty is the endpoint:__ Propagate joint parameter draws through covariance construction and optimization rather than stopping at coefficient intervals.

___
